In [ ]:
# Python environment used by pyadi-iio

pluto_python = raw"C:\Users\thoba\Documents\PlutoPythonTest\.venv\Scripts\python.exe"

ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = pluto_python

In [ ]:
# Packages

using PythonCall
using Statistics
using FFTW
using LinearAlgebra
using Plots
using LibSerialPort
using IJulia

plotly()

adi = pyimport("adi")
subprocess = pyimport("subprocess")

In [ ]:
# ============================================================
# Array settings
# ===========================================================

# HMC7992 switched RF paths
num_switch_paths = 4

# ============================================================
# STM32 settings
# ============================================================

stm_port = "COM8"
stm_baud = 115200

discard_buffers = 5

In [ ]:
# Connect to the STM32 over the USB serial port

stm = LibSerialPort.open(stm_port, stm_baud)

println("STM32 connected on ", stm_port)

In [ ]:
# Find the Pluto Ethernet address

result = subprocess.run(
    ["iio_info", "-s"],
    capture_output = true,
    text = true
)

output = pyconvert(String, result.stdout)

pluto_ip = nothing

for line in split(output, '\n')

    if occursin("Pluto", line)

        for match in eachmatch(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", line)

            ip = match.match

            # Ignore the Pluto USB-network address
            if ip != "192.168.2.1"
                pluto_ip = ip
                break
            end
        end
    end
end

pluto_ip === nothing &&
    error("Pluto Ethernet connection not found.")

pluto_uri = "ip:" * pluto_ip

sdr = adi.ad9361(uri = pluto_uri)

println("Pluto connected at ", pluto_ip)

In [ ]:
# Select one of the switched antenna paths

function select_path(stm, path)
    
    # Check that the requested path exists
    if path < 1 || path > num_switch_paths
        error("Invalid RF path: $path")
    end
    
    # Send the path number to the STM32
    command = string(path)
    write(stm, command)
    
    # Wait until the STM32 confirms the switched state
    response = strip(readline(stm))
    expected_response =  "READY $path"
    
    if response != expected_response
        error("Expected '$expected_response', received '$response'")
    end
    
    return nothing
end

In [ ]:
# ============================================================
# Cell 1 — Configure Pluto for RDF
# ============================================================


# RX1 = reference splitter branch
# RX2 = selected HMC7992 switch path


builtins = pyimport("builtins")
sdr.rx_enabled_channels = builtins.list((0, 1))

# Receiver settings
sdr.sample_rate = Int(6e6)
sdr.rx_rf_bandwidth = Int(5e6)
sdr.rx_lo = Int(867.8e6)

# Manual gain
sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"

sdr.rx_hardwaregain_chan0 = 20.0
sdr.rx_hardwaregain_chan1 = 20.0

# Clear old samples
for _ in 1:3
    sdr.rx()
end

println("Pluto configured")
println("Sample rate = ", sdr.sample_rate / 1e6, " MS/s")
println("RX LO = ", sdr.rx_lo / 1e6, " MHz")

In [ ]:
# ============================================================
# Cell 2 — Received signal detector
# ============================================================
function detect_vco(rx1, fc, fs)
    
    # Remove DC
    x = rx1 .- mean(rx1)
    
    N = length(x)
    
    # Hann window
    n = 0:N-1
    window = 0.5 .- 0.5 .* cos.(2π .* n ./ (N-1))
    
    # FFT
    X = fftshift(fft(x .* window))
        power = abs2.(X)
    
    # RF frequency axis
    freq = fc .+ (-N÷2 : N÷2 - 1) .* (fs / N )
    
    # Search expected VCO range
    search = (freq .>= 867.8e6) .& (freq .<= 868.8e6)
    
    search_indices = findall(search)
    
    # Strongest peak
    peak_index = search_indices[argmax(power[search])]
    
    f_vco = freq[peak_index]
    f_bb = f_vco - fc
    
    return f_vco, f_bb, freq, power
end

In [ ]:
# ============================================================
# Single received signal measurement
# ============================================================

# Read Pluto settings
fc = pyconvert(Float64, sdr.rx_lo)
fs = pyconvert(Float64, sdr.sample_rate)

# Discard one old capture
sdr.rx()

# Capture both RX channels
rx_data = sdr.rx()

# Use A1 / RX1 to detect the VCO
rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])

# Detect VCO
f_vco, f_bb, freq, power =
    detect_vco(rx1, fc, fs)

println(
    "Detected VCO = ",
    round(f_vco / 1e6, digits=6),
    " MHz"
)

println(
    "Baseband frequency = ",
    round(f_bb / 1e3, digits=2),
    " kHz"
)

# Convert power to dB for plotting
power_db = 10 .* log10.(power .+ 1e-20)

plot(
    freq ./ 1e6,
    power_db,
    xlabel = "RF Frequency (MHz)",
    ylabel = "Relative Power (dB)",
    title = "VCO Spectrum",
    legend = false,
    xlim = (867.8, 868.8)
)

vline!(
    [f_vco / 1e6],
    linestyle = :dash
)

In [ ]:
# ============================================================
# Measure fixed hardware phase for the selected RF path
# ============================================================

# Select the RF path to measure (1–4)
select_path(stm, 4)

# Clear old samples after switching
for _ in 1:3
    sdr.rx()
end

# Number of repeated measurements
num_measurements = 20

# Store phase from each capture
phase_measurements_deg = zeros(num_measurements)

for measurement in 1:num_measurements

    # Capture RX1 and RX2 together
    rx_data = sdr.rx()

    rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])
    rx2 = pyconvert(Vector{ComplexF64}, rx_data[1])

    N = length(rx1)
    n = 0:N-1

    # Hann window
    window = 0.5 .- 0.5 .* cos.(2π .* n ./ (N - 1))

    # Remove detected baseband frequency
    reference_tone =
        exp.(-1im .* 2π .* f_bb .* n ./ fs)

    # Complex value from each receiver
    V1 = sum(rx1 .* window .* reference_tone)
    V2 = sum(rx2 .* window .* reference_tone)

    # RX1 - RX2 phase
    phase_measurements_deg[measurement] =
        rad2deg(angle(V1 * conj(V2)))
end

# Circular mean
mean_phase_deg = rad2deg(angle(mean(exp.(1im .* deg2rad.(phase_measurements_deg)))))

# Phase spread around the mean
residuals_deg = rad2deg.(angle.(exp.(1im .* deg2rad.(phase_measurements_deg .- mean_phase_deg))))

phase_std_deg = std(residuals_deg)

println(
    "RF1 mean hardware phase = ",
    round(mean_phase_deg, digits=4),
    "°"
)

println(
    "RF1 phase standard deviation = ",
    round(phase_std_deg, digits=4),
    "°"
)

In [ ]:
# ============================================================
# Measure raw and hardware-corrected antenna phases
# with measurement stability checks
# ============================================================

hardware_phase_deg = [
    20.211,   # A1 - A2 / RF1
    21.669,   # A1 - A3 / RF2
    21.460,   # A1 - A4 / RF3
    21.116    # A1 - A5 / RF4
]

num_measurements = 20
settle_captures = 3

raw_phase_deg = zeros(4)
corrected_phase_deg = zeros(4)

phase_std_deg = zeros(4)
max_deviation_deg = zeros(4)

for path in 1:4

    # --------------------------------------------------------
    # Select RF1, RF2, RF3, RF4
    # --------------------------------------------------------

    select_path(stm, path)

    # Discard captures immediately after switching
    for _ in 1:settle_captures
        sdr.rx()
    end

    phase_measurements_deg = zeros(num_measurements)

    # --------------------------------------------------------
    # Take repeated phase measurements
    # --------------------------------------------------------

    for measurement in 1:num_measurements

        rx_data = sdr.rx()

        rx1 = pyconvert(
            Vector{ComplexF64},
            rx_data[0]
        )

        rx2 = pyconvert(
            Vector{ComplexF64},
            rx_data[1]
        )

        N = length(rx1)
        n = 0:N-1

        # Hann window
        window =
            0.5 .- 0.5 .* cos.(2π .* n ./ (N - 1))

        # Shift detected VCO tone to DC
        reference_tone =
            exp.(-1im .* 2π .* f_bb .* n ./ fs)

        # Complex tone values from both receivers
        V1 = sum(rx1 .* window .* reference_tone)
        V2 = sum(rx2 .* window .* reference_tone)

        # RX1 - RX2 phase
        phase_measurements_deg[measurement] =
            rad2deg(
                angle(V1 * conj(V2))
            )
    end

    # --------------------------------------------------------
    # Circular mean of the 20 measurements
    # --------------------------------------------------------

    raw_phase_deg[path] =
        rad2deg(
            angle(
                mean(
                    exp.(
                        1im .* deg2rad.(phase_measurements_deg)
                    )
                )
            )
        )

    # --------------------------------------------------------
    # Measure spread around circular mean
    # --------------------------------------------------------

    phase_errors_deg =
        rad2deg.(
            angle.(
                exp.(
                    1im .* deg2rad.(
                        phase_measurements_deg .-
                        raw_phase_deg[path]
                    )
                )
            )
        )

    phase_std_deg[path] =
        std(phase_errors_deg)

    max_deviation_deg[path] =
        maximum(abs.(phase_errors_deg))

    # --------------------------------------------------------
    # Hardware phase correction
    # --------------------------------------------------------

    corrected_phase_deg[path] =
        rad2deg(
            angle(
                exp(
                    1im * deg2rad(
                        raw_phase_deg[path] -
                        hardware_phase_deg[path]
                    )
                )
            )
        )

    # --------------------------------------------------------
    # Print result
    # --------------------------------------------------------

    println(
        "A1 - A$(path + 1)",
        "   raw = ",
        round(raw_phase_deg[path], digits = 2),
        "°   corrected = ",
        round(corrected_phase_deg[path], digits = 2),
        "°   std = ",
        round(phase_std_deg[path], digits = 2),
        "°   max dev = ",
        round(max_deviation_deg[path], digits = 2),
        "°"
    )
end

# ============================================================
# Overall quality check
# ============================================================

println()

if maximum(phase_std_deg) <= 2.0
    println("Measurement quality: GOOD")
else
    println("Measurement quality: CHECK / REPEAT")
end


In [ ]:
phase_270_deg = [
     107.04,
      50.02,
     -57.21,
    -119.56
]

phase_285_deg = [
     132.17,
     106.57,
     -11.45,
     -90.73
]

phase_300_deg = [
     134.48,
     158.97,
      32.48,
     -51.32
]

phase_315_deg = [
     128.71,
    -161.67,
      74.66,
      -8.47
]

phase_330_deg = [
     118.96,
    -137.32,
     119.40,
      22.10
]

phase_345_deg = [
     100.68,
    -135.36,
     170.05,
      49.00
]

phase_0_deg = [
      70.13,
    -144.39,
    -155.63,
      73.59
]

phase_255_deg = [
      85.14,
      -3.58,
    -115.70,
    -133.66
]

phase_240_deg = [
      50.85,
     -47.36,
    -157.99,
    -138.23
]

phase_225_deg = [
      15.34,
     -90.86,
     167.51,
    -132.91
]

phase_210_deg = [
     -38.24,
    -149.18,
     130.31,
    -137.70
]

phase_195_deg = [
     -72.55,
     169.70,
     118.74,
    -116.35
]

phase_180_deg = [
     -84.39,
     158.88,
     138.45,
     -83.37
]

phase_165_deg = [
    -103.44,
     154.69,
     163.96,
     -58.66
]

phase_150_deg = [
    -112.18,
     162.21,
    -147.27,
     -19.94
]

phase_135_deg = [
    -113.11,
    -176.27,
    -109.38,
      19.38
]

phase_120_deg = [
    -101.25,
    -137.93,
     -36.84,
      82.76
]

phase_105_deg = [
    -99.06,
    -98.82,
     15.16,
    116.36
]

phase_90_deg = [
    -94.58,
    -48.06,
     71.20,
    120.03
]

phase_75_deg = [
    -73.88,
      9.41,
    118.36,
    125.76
]

phase_60_deg = [
    -54.61,
     47.33,
   -175.82,
    125.15
]

phase_45_deg = [
    -33.73,
     79.90,
   -157.58,
    110.54
]

phase_30_deg = [
      6.40,
    135.33,
   -135.90,
    104.81
]

phase_15_deg = [
     49.16,
   -176.41,
   -149.16,
     89.29
]

In [ ]:
# ============================================================
# Open-space calibration measurements — 15° spacing
# ============================================================

cal_angles_deg = collect(0:15:345)

cal_measured_deg = [
     70.13  -144.39  -155.63    73.59
     49.16  -176.41  -149.16    89.29
      6.40   135.33  -135.90   104.81
    -33.73    79.90  -157.58   110.54
    -54.61    47.33  -175.82   125.15
    -73.88     9.41   118.36   125.76
    -94.58   -48.06    71.20   120.03
    -99.06   -98.82    15.16   116.36
   -101.25  -137.93   -36.84    82.76
   -113.11  -176.27  -109.38    19.38
   -112.18   162.21  -147.27   -19.94
   -103.44   154.69   163.96   -58.66
    -84.39   158.88   138.45   -83.37
    -72.55   169.70   118.74  -116.35
    -38.24  -149.18   130.31  -137.70
     15.34   -90.86   167.51  -132.91
     50.85   -47.36  -157.99  -138.23
     85.14    -3.58  -115.70  -133.66
    107.04    50.02   -57.21  -119.56
    132.17   106.57   -11.45   -90.73
    134.48   158.97    32.48   -51.32
    128.71  -161.67    74.66    -8.47
    118.96  -137.32   119.40    22.10
    100.68  -135.36   170.05    49.00
]

println("Calibration points = ", length(cal_angles_deg))
println("Matrix size = ", size(cal_measured_deg))

In [ ]:
# ============================================================
# Ideal UCA phase signatures at calibration angles
# ============================================================

c = 3e8
f0 = 868.1e6
λ = c / f0

Nant = 5
Rcircle = 0.09

A = zeros(Nant, 3)

for n in 1:Nant
    antenna_angle = 2π * (n - 1) / Nant

    A[n, 1] = Rcircle * cos(antenna_angle)
    A[n, 2] = Rcircle * sin(antenna_angle)
end

cal_ideal_deg = zeros(length(cal_angles_deg), 4)

for (i, theta_deg) in enumerate(cal_angles_deg)

    theta_rad = deg2rad(theta_deg)

    u = [
        cos(theta_rad),
        sin(theta_rad),
        0
    ]

    for n in 2:5

        path_difference =
            dot(u, A[1, :] - A[n, :])

        phase_deg =
            rad2deg(2π * path_difference / λ)

        # Wrap to -180° ... +180°
        cal_ideal_deg[i, n - 1] =
            rad2deg(
                angle(
                    exp(1im * deg2rad(phase_deg))
                )
            )
    end
end

println("Ideal phase matrix size = ", size(cal_ideal_deg))

cal_ideal_deg

In [ ]:
# ============================================================
# Array phase correction at the 24 calibration angles
# ============================================================

array_correction_deg =
    zeros(length(cal_angles_deg), 4)

for i in 1:length(cal_angles_deg)

    for j in 1:4

        difference =
            cal_measured_deg[i, j] -
            cal_ideal_deg[i, j]

        array_correction_deg[i, j] =
            rad2deg(
                angle(
                    exp(1im * deg2rad(difference))
                )
            )
    end
end

println(
    "Array correction matrix size = ",
    size(array_correction_deg)
)

array_correction_deg

In [ ]:
# ============================================================
# Circular interpolation of array correction: 0° to 359°
# ============================================================

scan_angles_deg = collect(0:359)

array_correction_interp_deg = zeros(360, 4)

for theta in scan_angles_deg

    # Calibration interval containing theta
    lower_index = floor(Int, theta / 15) + 1

    # Next calibration point, wrapping 345° -> 0°
    upper_index =
        lower_index == 24 ? 1 : lower_index + 1

    lower_angle =
        cal_angles_deg[lower_index]

    upper_angle =
        lower_angle + 15

    fraction =
        (theta - lower_angle) / 15

    for j in 1:4

        c1 =
            array_correction_deg[lower_index, j]

        c2 =
            array_correction_deg[upper_index, j]

        # Shortest angular change between calibration points
        correction_change =
            rad2deg(
                angle(
                    exp(
                        1im * deg2rad(c2 - c1)
                    )
                )
            )

        interpolated =
            c1 + fraction * correction_change

        # Wrap to -180° ... +180°
        array_correction_interp_deg[theta + 1, j] =
            rad2deg(
                angle(
                    exp(
                        1im * deg2rad(interpolated)
                    )
                )
            )
    end
end

println(
    "Interpolated correction matrix size = ",
    size(array_correction_interp_deg)
)

In [ ]:
# ============================================================
# Final calibrated phase lookup table: 0° to 359°
# ============================================================

ideal_phase_deg = zeros(360, 4)
calibrated_phase_deg = zeros(360, 4)

for theta in scan_angles_deg

    theta_rad = deg2rad(theta)

    u = [
        cos(theta_rad),
        sin(theta_rad),
        0
    ]

    for n in 2:5

        # Ideal geometrical phase
        path_difference =
            dot(u, A[1, :] - A[n, :])

        ideal =
            rad2deg(
                2π * path_difference / λ
            )

        ideal_phase_deg[theta + 1, n - 1] =
            rad2deg(
                angle(
                    exp(1im * deg2rad(ideal))
                )
            )

        # Ideal phase + interpolated array correction
        calibrated_phase_deg[theta + 1, n - 1] =
            rad2deg(
                angle(
                    exp(
                        1im * deg2rad(
                            ideal_phase_deg[theta + 1, n - 1] +
                            array_correction_interp_deg[theta + 1, n - 1]
                        )
                    )
                )
            )
    end
end

println(
    "Calibrated lookup matrix size = ",
    size(calibrated_phase_deg)
)

In [ ]:
# ============================================================
# Sanity check: calibrated lookup must reproduce
# all 24 calibration measurements
# ============================================================

for (i, theta) in enumerate(cal_angles_deg)

    model_phase =
        calibrated_phase_deg[theta + 1, :]

    measured_phase =
        cal_measured_deg[i, :]

    println(
        theta,
        "°  model = ",
        round.(model_phase, digits = 2),
        "   measured = ",
        round.(measured_phase, digits = 2)
    )
end

In [ ]:
validation_22_5_deg = [
     25.44,
    163.14,
   -133.91,
     93.55
]

validation_22_5_estimate_deg = 23
validation_22_5_rmse_deg = 4.64

validation_52_5_deg = [
    -42.26,
     66.47,
    178.93,
    106.18
]

validation_52_5_estimate_deg = 52
validation_52_5_rmse_deg = 10.17

validation_112_5_deg = [
    -102.50,
    -113.40,
      -4.05,
      96.06
]

validation_112_5_estimate_deg = 111
validation_112_5_rmse_deg = 3.99

validation_157_5_deg = [
    -101.46,
     167.17,
    -156.53,
     -27.25
]

validation_157_5_estimate_deg = 153
validation_157_5_rmse_deg = 6.07

validation_202_5_deg = [
     -39.20,
    -168.16,
     136.97,
    -121.11
]

validation_202_5_estimate_deg = 206
validation_202_5_rmse_deg = 9.66

validation_247_5_deg = [
     73.78,
    -20.79,
   -137.92,
   -133.48
]

validation_247_5_estimate_deg = 249
validation_247_5_rmse_deg = 2.79

validation_307_5_deg = [
     117.33,
    -168.60,
      59.45,
     -35.43
]

validation_307_5_estimate_deg = 309
validation_307_5_rmse_deg = 9.53

In [ ]:
# ============================================================
# Estimate DoA from four measured phases
# ============================================================

function estimate_doa(measured_phase_deg, calibrated_phase_deg)

    rmse_deg = zeros(360)

    for theta in 0:359

        phase_error_deg = zeros(4)

        for j in 1:4

            difference = measured_phase_deg[j] - calibrated_phase_deg[theta + 1, j]

            phase_error_deg[j] = rad2deg(angle(exp(1im * deg2rad(difference))))
        end

        rmse_deg[theta + 1] = sqrt(mean(phase_error_deg .^ 2))
    end

    estimated_angle = argmin(rmse_deg) - 1

    return estimated_angle, rmse_deg
end

In [ ]:
measured_phase_deg = [
    146.69,
     77.62,
    -19.97,
    -95.41
]

estimated_angle, rmse_deg =
    estimate_doa(measured_phase_deg, calibrated_phase_deg)

println("Estimated DoA = ", estimated_angle, "°")
println("Minimum RMSE = ", round(minimum(rmse_deg), digits=2), "°")

In [ ]:
# ============================================================
# INDOOR VALIDATION RESULTS
# ============================================================

# Test 1: Nominal transmitter angle = 22.5°

indoor_22_5_deg = [
     24.95,
    144.88,
   -166.77,
     87.59
]

indoor_22_5_estimate_deg = 24
indoor_22_5_rmse_deg = 15.72

# Phase measurement stability
indoor_22_5_std_deg = [
    0.18,
    0.24,
    0.10,
    0.07
]


# Test 2: Nominal transmitter angle = 52.5°

indoor_52_5_deg = [
    -29.23,
     43.03,
    177.37,
     97.46
]

indoor_52_5_estimate_deg = 56
indoor_52_5_rmse_deg = 18.31

indoor_52_5_std_deg = [
    0.49,
    0.25,
    0.19,
    0.13
]

# Test 3: Nominal transmitter angle = 112.5°

indoor_112_5_deg = [
    -91.36,
   -101.05,
    -22.59,
    101.60
]

indoor_112_5_estimate_deg = 112
indoor_112_5_rmse_deg = 11.84

indoor_112_5_std_deg = [
    0.17,
    0.30,
    0.18,
    0.25
]

# Test 4: Nominal transmitter angle = 157.5°

indoor_157_5_deg = [
   -102.85,
    178.62,
   -142.33,
    -46.04
]

indoor_157_5_estimate_deg = 152
indoor_157_5_rmse_deg = 15.64

indoor_157_5_std_deg = [
    0.35,
    0.16,
    0.09,
    0.14
]

# Test 5: Nominal transmitter angle = 202.5°

indoor_202_5_deg = [
    -58.24,
   -147.92,
    142.23,
   -127.09
]

indoor_202_5_estimate_deg = 207
indoor_202_5_rmse_deg = 11.75

indoor_202_5_std_deg = [
    0.13,
    0.21,
    0.13,
    0.12
]

# Test 6: Nominal transmitter angle = 247.5°
# Third measurement

indoor_247_5_deg = [
     60.18,
    -24.65,
   -135.17,
    -91.39
]

indoor_247_5_estimate_deg = 248
indoor_247_5_rmse_deg = 23.15

indoor_247_5_std_deg = [
    0.46,
    0.24,
    0.07,
    0.30
]

indoor_307_5_deg = [
    126.95,
    146.36,
     34.66,
    -61.80
]

indoor_307_5_estimate_deg = 298
indoor_307_5_rmse_deg = 6.78

indoor_307_5_std_deg = [
    0.64,
    0.13,
    0.32,
    0.13
]

# Additional Indoor Test: Nominal transmitter angle = 180°

indoor_180_deg = [
    -68.75,
    150.87,
    135.61,
   -104.06
]

indoor_180_estimate_deg = 187
indoor_180_rmse_deg = 9.10

# Additional Indoor Test: Nominal transmitter angle = 90°

indoor_90_deg = [
    -97.51,
    -45.14,
     57.88,
    154.43
]

indoor_90_estimate_deg = 91
indoor_90_rmse_deg = 18.23

indoor_90_std_deg = [
    0.06,
    0.07,
    0.08,
    0.28
]

# Additional Indoor Test: Nominal transmitter angle = 0°

indoor_0_deg = [
     69.17,
   -148.75,
   -158.39,
     90.65
]

indoor_0_estimate_deg = 4
indoor_0_rmse_deg = 7.30

indoor_0_std_deg = [
    0.07,
    0.14,
    0.16,
    0.07
]

In [ ]:
repeatability_phases = [
    140.20  67.77  -25.66  -83.12
    140.03  67.93  -25.64  -83.24
    139.92  67.47  -25.94  -83.90
    139.89  67.75  -25.61  -83.30
    139.89  67.58  -25.63  -83.47
    139.33  67.40  -26.29  -84.18
    139.49  67.73  -25.55  -83.08
    139.67  67.39  -25.94  -83.83
    139.44  67.28  -26.00  -83.76
    139.30  67.34  -25.99  -83.79
]

repeatability_estimates = Float64[]
repeatability_rmse = Float64[]

for k in axes(repeatability_phases, 1)

    angle, rmse_deg = estimate_doa(
        vec(repeatability_phases[k, :]),
        calibrated_phase_deg
    )

    push!(repeatability_estimates, angle)
    push!(repeatability_rmse, minimum(rmse_deg))

    println(
        "Trial $k: DoA = $(angle)°, " *
        "RMSE = $(round(minimum(rmse_deg), digits=2))°"
    )
end

In [ ]:
repeatability_180_phases = [
    -79.91  -166.67  160.35  -127.44
    -78.63  -165.82  159.94  -126.92
    -78.45  -165.87  162.14  -126.60
    -78.66  -165.63  159.26  -127.68
    -78.72  -165.85  160.94  -127.17
    -78.70  -165.93  162.81  -126.49
    -78.90  -166.02  159.01  -127.38
    -78.79  -165.95  159.19  -127.30
    -78.59  -165.81  159.06  -127.20
    -79.12  -166.15  159.19  -127.64
]

repeatability_180_estimates = Float64[]
repeatability_180_rmse = Float64[]

for k in axes(repeatability_180_phases, 1)

    angle, rmse_deg = estimate_doa(
        vec(repeatability_180_phases[k, :]),
        calibrated_phase_deg
    )

    push!(repeatability_180_estimates, angle)
    push!(repeatability_180_rmse, minimum(rmse_deg))

    println(
        "Trial $k: DoA = $(angle)°, " *
        "RMSE = $(round(minimum(rmse_deg), digits=2))°"
    )
end


In [ ]:
using Plots
gr()
phase_labels = [
    "Δϕ₁₂",
    "Δϕ₁₃",
    "Δϕ₁₄",
    "Δϕ₁₅"
]

plots = []

for j in 1:4

    pj = plot(
        scan_angles_deg,
        array_correction_interp_deg[:, j],
        xlabel = "Azimuth angle (°)",
        ylabel = "Phase correction (°)",
        title = phase_labels[j],
        xlim = (0, 359),
        xticks = 0:90:270,
        linewidth = 2,
        label = "Interpolated correction",
        legend = :bottomright
    )

    scatter!(
        pj,
        cal_angles_deg,
        array_correction_deg[:, j],
        markersize = 4,
        label = "Measured points"
    )

    push!(plots, pj)
end

p = plot(
    plots...,
    layout = (2, 2),
    size = (950, 700)
)

display(p)

savefig(p, "array_phase_correction_interpolation_subplots.png")